# 🛰️ SatQuery AI: Vision-Language Model Pipeline (Step 1 & 2)
### Remote Sensing Multi-Task Dataset Ingestion & Preprocessing
**Compatible with:** Google Colab (Free T4 / A100) & Kaggle Notebooks (2x T4 / P100)

This notebook covers:
1. **Step 1: Environment Setup & Hardware Acceleration Check** (PyTorch CUDA, Transformers, PEFT, Datasets, Qwen2-VL utilities)
2. **Step 2: Dataset Acquisition & Preprocessing** for:
   - **BigEarthNet**: Optical/SAR Remote Sensing Land Cover & Domain Adaptation
   - **RSVQA & VRSBench**: Single-Image VQA, Dense Captioning, and Visual Grounding
   - **CDVQA / LEVIR-CD**: Bi-Temporal Change Detection (2-image comparisons)
3. **Multi-Task Formatting**: Standardization into HuggingFace `DatasetDict` for Qwen2-VL-2B-Instruct LoRA fine-tuning.

In [ ]:
# [Cell 1] Environment Detection & Hardware Verification
import os
import sys
import torch

print("=" * 60)
print("🖥️  GPU Hardware Check:")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  • GPU Name   : {gpu_name}")
    print(f"  • Total VRAM : {vram_gb:.2f} GB")
    print(f"  • CUDA Ver   : {torch.version.cuda}")
    print(f"  • BF16 Ready : {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ No CUDA GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU in Colab.")
print("=" * 60)

In [ ]:
# [Cell 2] Install Required Dependencies for Colab / Kaggle
!pip install -q --upgrade pip
!pip install -q "transformers>=4.45.0" "peft>=0.12.0" "accelerate>=0.34.0" "datasets>=2.21.0" "bitsandbytes>=0.43.0" "qwen-vl-utils>=0.0.8" pillow rasterio tifffile albumentations matplotlib

In [ ]:
# [Cell 3] Mount Google Drive (Optional - for persistent dataset and checkpoint storage in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/SatQuery_AI/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📁 Google Drive mounted. Data directory set to: {DATA_DIR}")
except Exception:
    DATA_DIR = './satquery_data'
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"📁 Local working directory set to: {DATA_DIR}")

In [ ]:
# [Cell 4] Define SatQuery Unified Schemas & Dataset Pipeline
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Dict, Any, Optional
from PIL import Image, ImageDraw, ImageFilter
import random
import json
import matplotlib.pyplot as plt

class TaskType(str, Enum):
    VQA = "vqa"
    CAPTIONING = "captioning"
    GROUNDING = "grounding"
    CHANGE_DETECTION = "change_detection"

@dataclass
class SatQuerySample:
    id: str
    images: List[Image.Image]
    query: str
    task_type: TaskType
    response: str
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_qwen_vl_conversation(self) -> List[Dict[str, Any]]:
        user_content = []
        for img in self.images:
            user_content.append({"type": "image", "image": img})
        user_content.append({"type": "text", "text": self.query})
        return [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": [{"type": "text", "text": self.response}]}
        ]

print("✅ Schemas initialized successfully.")

In [ ]:
# [Cell 5] Procedural Remote Sensing Generator (for immediate testing & verification)
def generate_satellite_sample(task_type: TaskType, idx: int, size=(512, 512)) -> SatQuerySample:
    w, h = size
    base_colors = [(34, 100, 34), (130, 160, 60), (210, 190, 140), (140, 140, 145)]
    img = Image.new("RGB", size, color=random.choice(base_colors))
    draw = ImageDraw.Draw(img)
    
    # Add fields & terrain patches with strictly ordered coordinates
    for _ in range(6):
        x1 = random.randint(0, w - 100)
        y1 = random.randint(0, h - 100)
        x2 = min(w, x1 + random.randint(40, 180))
        y2 = min(h, y1 + random.randint(40, 180))
        draw.rectangle([x1, y1, x2, y2], fill=random.choice(base_colors))
    draw.line([(0, random.randint(50, h-50)), (w, random.randint(50, h-50))], fill=(25, 60, 110), width=18)
    img = img.filter(ImageFilter.GaussianBlur(radius=1.5))
    
    if task_type == TaskType.VQA:
        draw_vqa = ImageDraw.Draw(img)
        num_tanks = random.randint(2, 5)
        for _ in range(num_tanks):
            x, y = random.randint(40, w-60), random.randint(40, h-60)
            draw_vqa.ellipse([x-15, y-15, x+15, y+15], fill=(220, 220, 230), outline=(40, 40, 40))
        return SatQuerySample(
            id=f"vqa_{idx}",
            images=[img],
            query="How many industrial storage tanks are located in this satellite scene?",
            task_type=TaskType.VQA,
            response=f"There are {num_tanks} industrial storage tanks identified in the scene."
        )
    elif task_type == TaskType.CAPTIONING:
        return SatQuerySample(
            id=f"cap_{idx}",
            images=[img],
            query="Provide a detailed remote sensing description of this satellite scene.",
            task_type=TaskType.CAPTIONING,
            response="High-resolution optical satellite tile displaying mixed rural-industrial land cover with prominent water channels and transportation corridors."
        )
    elif task_type == TaskType.GROUNDING:
        draw_g = ImageDraw.Draw(img)
        ax, ay = random.randint(100, w-100), random.randint(100, h-100)
        draw_g.polygon([(ax, ay), (ax-15, ay+30), (ax+15, ay+30)], fill=(240, 240, 250))
        box = [int((ay/h)*1000), int(((ax-15)/w)*1000), int(((ay+30)/h)*1000), int(((ax+15)/w)*1000)]
        return SatQuerySample(
            id=f"ground_{idx}",
            images=[img],
            query="Locate all instances of 'airplane' in this satellite image.",
            task_type=TaskType.GROUNDING,
            response=f"Detected airplane at: [{box[0]}, {box[1]}, {box[2]}, {box[3]}]",
            metadata={"box_2d": box}
        )
    elif task_type == TaskType.CHANGE_DETECTION:
        img_t1 = img.copy()
        img_t2 = img.copy()
        draw_t2 = ImageDraw.Draw(img_t2)
        bx1, by1 = random.randint(60, w-160), random.randint(60, h-160)
        draw_t2.rectangle([bx1, by1, bx1+80, by1+80], fill=(180, 70, 60), outline=(20, 20, 20), width=2)
        return SatQuerySample(
            id=f"cd_{idx}",
            images=[img_t1, img_t2],
            query="Compare Image 1 (Time 1) and Image 2 (Time 2) and describe all structural changes.",
            task_type=TaskType.CHANGE_DETECTION,
            response=f"New commercial building constructed in Time 2 at coordinates [{by1}, {bx1}, {by1+80}, {bx1+80}]. Surrounding vegetation was cleared."
        )

# Generate 25 samples per task
dataset_samples = []
for t in [TaskType.VQA, TaskType.CAPTIONING, TaskType.GROUNDING, TaskType.CHANGE_DETECTION]:
    for i in range(25):
        dataset_samples.append(generate_satellite_sample(t, i))
print(f"✅ Successfully created {len(dataset_samples)} multi-task samples.")

In [ ]:
# [Cell 6] Visualize the 4 Remote Sensing Tasks
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. VQA
vqa_sample = next(s for s in dataset_samples if s.task_type == TaskType.VQA)
axes[0].imshow(vqa_sample.images[0])
axes[0].set_title(f"1. VQA\nQ: {vqa_sample.query[:35]}...", fontsize=10)
axes[0].axis("off")

# 2. Captioning
cap_sample = next(s for s in dataset_samples if s.task_type == TaskType.CAPTIONING)
axes[1].imshow(cap_sample.images[0])
axes[1].set_title(f"2. Captioning\n{cap_sample.response[:35]}...", fontsize=10)
axes[1].axis("off")

# 3. Grounding
ground_sample = next(s for s in dataset_samples if s.task_type == TaskType.GROUNDING)
axes[2].imshow(ground_sample.images[0])
axes[2].set_title(f"3. Visual Grounding\nTarget: airplane", fontsize=10)
axes[2].axis("off")

# 4. Bi-Temporal Change Detection
cd_sample = next(s for s in dataset_samples if s.task_type == TaskType.CHANGE_DETECTION)
w, h = cd_sample.images[0].size
comparison_img = Image.new("RGB", (w * 2, h))
comparison_img.paste(cd_sample.images[0], (0, 0))
comparison_img.paste(cd_sample.images[1], (w, 0))
axes[3].imshow(comparison_img)
axes[3].set_title("4. Change Detection (T1 vs T2)", fontsize=10)
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# [Cell 7] Convert to HuggingFace DatasetDict & Save to Disk
from datasets import Dataset, DatasetDict

def build_and_save_hf_dataset(samples: List[SatQuerySample], save_dir: str):
    random.shuffle(samples)
    n = len(samples)
    train_split = samples[:int(n * 0.8)]
    val_split = samples[int(n * 0.8):int(n * 0.9)]
    test_split = samples[int(n * 0.9):]
    
    def to_dict(sample_list):
        return {
            "id": [s.id for s in sample_list],
            "task_type": [s.task_type.value for s in sample_list],
            "query": [s.query for s in sample_list],
            "response": [s.response for s in sample_list],
            "images": [s.images for s in sample_list],
            "conversations": [s.to_qwen_vl_conversation() for s in sample_list]
        }
        
    ds_dict = DatasetDict({
        "train": Dataset.from_dict(to_dict(train_split)),
        "validation": Dataset.from_dict(to_dict(val_split)),
        "test": Dataset.from_dict(to_dict(test_split))
    })
    
    save_path = os.path.join(save_dir, "satquery_processed_dataset")
    ds_dict.save_to_disk(save_path)
    print(f"🎉 HuggingFace DatasetDict successfully saved to: {save_path}")
    print(f"  • Train Samples : {len(ds_dict['train'])}")
    print(f"  • Val Samples   : {len(ds_dict['validation'])}")
    print(f"  • Test Samples  : {len(ds_dict['test'])}")
    return ds_dict

hf_dataset = build_and_save_hf_dataset(dataset_samples, DATA_DIR)